# 🕹️ <span style=color:dodgerblue>LLM for video game knowledge assistance<span>

This notebook is used for explanation purpose. Once the `docker compose up` command
is executed the entire application is ready to be used.

## 📔 <span style=color:gold>What is this notebook about?</span>
This notebook is intended to explain all the pipeline and provide an overview 
of the processes involved in the application (ingestion, monitoring, evaluation, etc).  
It's a complement to the [README.md](README.md) that shows the internal mechanisms
of the scripts.

In [ ]:
from llm import RAGClient
from opensearchpy import OpenSearch

## <span style=color:green>📂 Opensearch client creation</span>

We create an opensearch client. It is used for indexation and search 
(lexical, semantic, hybrid).  
The `RAGClient` simplifies the use of the RAG.

In [ ]:
opensearch_client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "Opensearch16admin#"),
    use_ssl=False,
    verify_certs=False, 
    ssl_show_warn=False,
)

rag_client = RAGClient(opensearch_client)

## <span style=color:lightsalmon>⛁ Ingestion</span>

This part showcases the ingestion process and how It works. To avoid the long 
waiting time this part is entirely optional and is not necessary to proceed to 
the blocks of this notebook It's just used for explanation purpose.

In [ ]:
from ingest import IGDB, Wikipedia
from opensearch_utils import setup_embedder

# There is already a pre setup embedder, you don't need
#setup_embedder(opensearch_client, model="huggingface/sentence-transformer")

### 🎮 <span style=color:darkorchid>IGDB ingestion</span>

**You can skip this section if you want.**

> <span style=color:yellow>⚠️ **Warning**</span>    
> If you want to execute the IGDB ingestion yourself, you must have and IGDB 
> developer key and an account.  
> Please follow the instructions in this link: https://api-docs.igdb.com/#getting-started.  
> Then write all your information in the `.env` file.

In [ ]:
igdb = IGDB(opensearch_client)
igdb.download(index="igdb_small")

### <span style=color:deepskyblue>📄 Wikipedia ingestion</span>
We pull the wikipedia information from huggingface index. The data was updated 
on 


In [ ]:
wikipedia = Wikipedia(opensearch_client)
wikipedia.download(index="wikipedia_small")

## 🧪 <span style=color:forestgreen>Evaluation</span>

Here we will use our `Evaluator` class to execute each one of the steps:
1. Ground truth generation
2. Search evaluation & optimization.
    1. Perform the base evaluation itself.
    2. Optimize the boost values.
3. Tools use and final RAG's answer evaluation.

In [ ]:
import pandas as pd
from evaluation import Evaluator

evaluator = Evaluator(rag_client)

### 🎯 <span style=color:orangered>Ground truth generation</span>

In order to evaluate the search quality and model's performance we must have 
querys and a target variable to use as evaluation method.  
In this case, the target variable is the document id and the query will be a llm 
generated question based on the target document. 

For instance, if the target document is about Mario Kart, the llm
will generate `n` questions related to the document topic.

> <span style=color:yellow>⚠️ **Warning**</span>  
> The code blocks in this section create a very small dataset so as to
> not waste your tokens. You may create a very large dataset if you want

In [ ]:
wikipedia_ground_truth_small = evaluator.generate_ground_truth(
    index="wikipedia",
    count=3,
    n=3,
    file_path="data/wikipedia_ground_truth_small.csv",
)

igdb_ground_truth_small = evaluator.generate_ground_truth(
    index="igdb",
    count=3,
    n=3,
    file_path="data/igdb_ground_truth_small.csv",
)

### 🔎 <span style=color:goldenrod>Search evaluation & optimization</span>

We must evaluate our search functions. Do they return the relevant documents?
For this we will use two metrics

In [ ]:
igdb_hr_score, igdb_mrr_score, igdb_x = evaluator.evaluate_search(index="igdb")

wikipedia_hr_score, wikipedia_mrr_score, wikipedia_x = evaluator.evaluate_search(
    index="wikipedia"
)

display(
    "--- IGDB search evaluation ---"
    f"Hit rate score: {igdb_hr_score}"
    f"Mean reciprocal rank score: {igdb_mrr_score}"
    "--- wikipedia search evaluation ---"
    f"Hit rate score: {wikipedia_hr_score}"
    f"Mean reciprocal rank score: {wikipedia_mrr_score}"
)

Now we want to use a boost dict. The 
Here our method is very naive, we will only test in a given set of values (n 
dimensional grid) and take the combination that returns the best result.

In [ ]:
# Search boosting optimization

### 🔧 <span style=color:silver>Tools and final RAG answer evaluation</span>

Here is the final evaluation. We want to see the performance of our agent when
using the entire RAG system. So we use two sources of evaluation. The user's 
evaluation and a separate llm-judge evaluation. The judge will evaluate the tool
usage and the final answer quality based on the ground truth. The user only 
evaluates the final answer with good or bad review.

In [ ]:
judge = RAGClient(opensearch_client, model="gemma-4-31b-it")
evaluator.evaluate_agent(judge)

## 👁️ <span style=color:gold>Monitoring</span>

The final step is the monitoring, we must see the **usage** (tokens and cost) 
and **performance** (reviews) of our agent.  
For this, we can open the *streamlit* app in this address http://localhost:8501,
Click to the 